De nuestra actividad pasada, se retoma el esfuerzo en el que se realizo el pipeline de preprocesamiento para los conjuntos de entrenamiento, validacion y prueba

In [ ]:
from sklearn.preprocessing import FunctionTransformer


pipeline_num_log = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("log1p", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
    ("scaler", StandardScaler())
])

pipeline_num_sin_log = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

pipeline_cat = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")) # drop="first"
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num_log", pipeline_num_log, variables_log),
        ("num", pipeline_num_sin_log, variables_num_sin_log),
        ("cat", pipeline_cat, categoricas)
    ],
    remainder="drop"
)

In [ ]:
preprocessor.fit(Xtrain)

XtrainFit = preprocessor.transform(Xtrain)
XvalFit = preprocessor.transform(Xval)
XtestFit = preprocessor.transform(Xtest)

Ya una vez definidos los conjuntos procedemos a la ejecucion de los modelos

### 1. Lasso Regressor

In [445]:
from sklearn.linear_model import Lasso

In [450]:
param_grid = {
    'alpha':[0.01,0.02,0.03]
}

lasso = Lasso(
    random_state=42
)

grid_search_lasso = GridSearchCV(estimator=lasso, param_grid=param_grid, cv=5, scoring='neg_mean_squared_error')

In [451]:
grid_search_lasso.fit(XtrainFit, ytrain)

pred_train_lasso_log = grid_search_lasso.predict(XtrainFit)

pred_val_lasso_log = grid_search_lasso.predict(XvalFit)

In [452]:
mae_train_lasso_log = mean_absolute_error(
    ytrain,
    pred_train_lasso_log
)

rmse_train_lasso_log = np.sqrt(mean_squared_error(
    ytrain, 
    pred_train_lasso_log)
)

r2_train_lasso_log = r2_score(
    ytrain,
    pred_train_lasso_log
)

#Val
mae_val_lasso_log = mean_absolute_error(
    yval,
    pred_val_dt_log
)

rmse_val_lasso_log = np.sqrt(mean_squared_error(
    yval, 
    pred_val_lasso_log)
)

r2_val_lasso_log = r2_score(
    yval,
    pred_val_lasso_log
)

In [453]:
print("Lasso Regression escala logaritmica")

print("\nTRAIN")
print("MAE log:", mae_train_lasso_log)
print("RMSE log:", rmse_train_lasso_log)
print("R2 log:", r2_train_lasso_log)

print("\nVALIDATION")
print("MAE log:", mae_val_lasso_log)
print("RMSE log:", rmse_val_lasso_log)
print("R2 log:", r2_val_lasso_log)

Lasso Regression escala logaritmica

TRAIN
MAE log: 0.41394460164263835
RMSE log: 0.5531166754633092
R2 log: 0.17057232688764934

VALIDATION
MAE log: 0.40958819826459064
RMSE log: 0.5493167180731707
R2 log: 0.17965859356765423


### 2. Decision Tree Regressor

In [412]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import GridSearchCV

dtree = DecisionTreeRegressor(random_state=42)


param_grid = {
    # 'criterion': ['squared_error', 'friedman_mse', 'absolute_error'],
    'max_depth': [None, 5, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

grid_search_dt = GridSearchCV(estimator=dtree, param_grid=param_grid, cv=5, scoring='neg_mean_squared_error')

In [413]:
grid_search_dt.fit(XtrainFit, ytrain)

pred_train_dt_log = grid_search_dt.predict(XtrainFit)

pred_val_dt_log = grid_search_dt.predict(XvalFit)

In [414]:
mae_train_dt_log = mean_absolute_error(
    ytrain,
    pred_train_dt_log
)

rmse_train_dt_log = np.sqrt(mean_squared_error(
    ytrain, 
    pred_train_dt_log)
)

r2_train_dt_log = r2_score(
    ytrain,
    pred_train_dt_log
)

#Val
mae_val_dt_log = mean_absolute_error(
    yval,
    pred_val_dt_log
)

rmse_val_dt_log = np.sqrt(mean_squared_error(
    yval, 
    pred_val_dt_log)
)

r2_val_dt_log = r2_score(
    yval,
    pred_val_dt_log
)

In [415]:
print("Decision Tree escala logaritmica")

print("\nTRAIN")
print("MAE log:", mae_train_dt_log)
print("RMSE log:", rmse_train_dt_log)
print("R2 log:", r2_train_dt_log)

print("\nVALIDATION")
print("MAE log:", mae_val_dt_log)
print("RMSE log:", rmse_val_dt_log)
print("R2 log:", r2_val_dt_log)

Decision Tree escala logaritmica

TRAIN
MAE log: 0.40351141008936037
RMSE log: 0.5344550659221575
R2 log: 0.22559631067247765

VALIDATION
MAE log: 0.40958819826459064
RMSE log: 0.5438771568177725
R2 log: 0.1958248722480438


### 3. KNN Regressor

In [432]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import GridSearchCV

# 1. Initialize the estimator
knn = KNeighborsRegressor()

# 2. Define the parameter grid
param_grid = {
    'n_neighbors': [5, 10, 15, 20, 30, 35,40],
    # 'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']
}

# 3. Setup GridSearchCV
# cv=5 performs 5-fold cross-validation
grid_search_knn = GridSearchCV(estimator=knn, param_grid=param_grid, cv=5, scoring='neg_mean_squared_error')

In [433]:
grid_search_knn.fit(XtrainFit, ytrain)

pred_train_knn_log = grid_search_knn.predict(XtrainFit)

pred_val_knn_log = grid_search_knn.predict(XvalFit)

In [435]:
mae_train_knn_log = mean_absolute_error(
    ytrain,
    pred_train_knn_log
)

rmse_train_knn_log = np.sqrt(mean_squared_error(
    ytrain, 
    pred_train_knn_log)
)

r2_train_knn_log = r2_score(
    ytrain,
    pred_train_knn_log
)

#Val
mae_val_knn_log = mean_absolute_error(
    yval,
    pred_val_knn_log
)

rmse_val_knn_log = np.sqrt(mean_squared_error(
    yval, 
    pred_val_knn_log)
)

r2_val_knn_log = r2_score(
    yval,
    pred_val_knn_log
)

In [437]:
print("KNN escala logaritmica")

print("\nTRAIN")
print("MAE log:", mae_train_knn_log)
print("RMSE log:", rmse_train_knn_log)
print("R2 log:", r2_train_knn_log)

print("\nVALIDATION")
print("MAE log:", mae_val_knn_log)
print("RMSE log:", rmse_val_knn_log)
print("R2 log:", r2_val_knn_log)

KNN escala logaritmica

TRAIN
MAE log: 0.38927469718951185
RMSE log: 0.5230888659814982
R2 log: 0.25818438711801406

VALIDATION
MAE log: 0.398285691731231
RMSE log: 0.5311935353524642
R2 log: 0.23289544486637426


### 4. Support Vector Regressor

In [513]:
from sklearn.svm import SVR

svr=SVR()
param_grid = {
    # 'kernel': ['rbf', 'linear'],
    'C': [0.1, 0.4,0.5],
    'gamma': [0.01, 0.1],
    # 'epsilon': [0.01, 0.1, 0.2]
}


grid_search_svr = GridSearchCV(
    estimator=svr,
    param_grid=param_grid,
    cv=5,
    scoring='neg_mean_squared_error',
)

In [514]:
grid_search_svr.fit(XtrainFit, ytrain)

pred_train_svr_log = grid_search_svr.predict(XtrainFit)

pred_val_svr_log = grid_search_svr.predict(XvalFit)

KeyboardInterrupt: 

### 5. Ridge Polinomica

In [465]:
from sklearn.preprocessing import PolynomialFeatures

In [504]:
pipe = Pipeline([
    ("poly", PolynomialFeatures(include_bias=False)),
    ("model", Ridge())
])

param_grid = {
    'model__alpha':[0.01,0.02,0.05,0.2,0.5,1,2]
    ,'poly__degree': [2]
    ,'poly__interaction_only': [False]
}


grid_search_ridge_poly = GridSearchCV(pipe, param_grid=param_grid, cv=5, scoring='neg_mean_squared_error')

In [505]:
grid_search_ridge_poly.fit(XtrainFit, ytrain)

pred_train_ridge_poly_log = grid_search_ridge_poly.predict(XtrainFit)

pred_val_ridge_poly_log = grid_search_ridge_poly.predict(XvalFit)

In [506]:
mae_train_ridge_poly_log = mean_absolute_error(
    ytrain,
    pred_train_ridge_poly_log
)

rmse_train_ridge_poly_log = np.sqrt(mean_squared_error(
    ytrain, 
    pred_train_ridge_poly_log)
)

r2_train_ridge_poly_log = r2_score(
    ytrain,
    pred_train_ridge_poly_log
)

#Val
mae_val_ridge_poly_log = mean_absolute_error(
    yval,
    pred_val_ridge_poly_log
)

rmse_val_ridge_poly_log = np.sqrt(mean_squared_error(
    yval, 
    pred_val_ridge_poly_log)
)

r2_val_ridge_poly_log = r2_score(
    yval,
    pred_val_ridge_poly_log
)

In [508]:
print("Ridge Poly escala logaritmica")

print("\nTRAIN")
print("MAE log:", mae_train_ridge_poly_log)
print("RMSE log:", rmse_train_ridge_poly_log)
print("R2 log:", r2_train_ridge_poly_log)

print("\nVALIDATION")
print("MAE log:", mae_val_ridge_poly_log)
print("RMSE log:", rmse_val_ridge_poly_log)
print("R2 log:", r2_val_ridge_poly_log)

Ridge Poly escala logaritmica

TRAIN
MAE log: 0.36856597784404416
RMSE log: 0.4925455286124179
R2 log: 0.3422849507745094

VALIDATION
MAE log: 0.3899937156032036
RMSE log: 0.5214125463657082
R2 log: 0.2608851062506228


### 6. Lasso Polinomica

In [509]:
pipe = Pipeline([
    ("poly", PolynomialFeatures(include_bias=False)),
    ("model", Lasso())
])

param_grid = {
    'model__alpha':[0.01,0.02,0.05,0.2,0.5,1,2]
    ,'poly__degree': [2]
    ,'poly__interaction_only': [False]
}


grid_search_lasso_poly = GridSearchCV(pipe, param_grid=param_grid, cv=5, scoring='neg_mean_squared_error')

In [510]:
grid_search_lasso_poly.fit(XtrainFit, ytrain)

pred_train_lasso_poly_log = grid_search_lasso_poly.predict(XtrainFit)

pred_val_lasso_poly_log = grid_search_lasso_poly.predict(XvalFit)

In [511]:
mae_train_lasso_poly_log = mean_absolute_error(
    ytrain,
    pred_train_lasso_poly_log
)

rmse_train_lasso_poly_log = np.sqrt(mean_squared_error(
    ytrain, 
    pred_train_lasso_poly_log)
)

r2_train_lasso_poly_log = r2_score(
    ytrain,
    pred_train_lasso_poly_log
)

#Val
mae_val_lasso_poly_log = mean_absolute_error(
    yval,
    pred_val_lasso_poly_log
)

rmse_val_lasso_poly_log = np.sqrt(mean_squared_error(
    yval, 
    pred_val_lasso_poly_log)
)

r2_val_lasso_poly_log = r2_score(
    yval,
    pred_val_lasso_poly_log
)

In [512]:
print("Lasso Poly escala logaritmica")

print("\nTRAIN")
print("MAE log:", mae_train_lasso_poly_log)
print("RMSE log:", rmse_train_lasso_poly_log)
print("R2 log:", r2_train_lasso_poly_log)

print("\nVALIDATION")
print("MAE log:", mae_val_lasso_poly_log)
print("RMSE log:", rmse_val_lasso_poly_log)
print("R2 log:", r2_val_lasso_poly_log)

Lasso Poly escala logaritmica

TRAIN
MAE log: 0.3966814404921481
RMSE log: 0.5292627688124991
R2 log: 0.24057007500567062

VALIDATION
MAE log: 0.39657404824696596
RMSE log: 0.5273880562012639
R2 log: 0.24384717374919407


### Mejores dos modelos

Se eligen los modelos de Regresion Polinomina Lasso y KNNRegressor por sus porcentajes superiores de R2. Dentro del tradeoff para la eleccion de los 2 mejores modelos, el factor del tiempo de procesamiento juega e influye, sobre todo para el modelo de Support Vector Regressor, el cual se descarto el aanlisis de su performance al presentar tiempos elevados de procesamiento

El modelo elegido sera el modelo Regression Polinomica Lasso, debido a que presenta porcentajes de R2 superiores a los otros modelos, asi como no presentar sobreajuste por tener porcentajes iguales en la poblacion de entrenamieto y validacion

El ajuste secundario no se realiza debido a que ya se hizo el ajuste por gridsearchcv